# 04 — mini-GPT: Text Generation from Scratch

> **Goal:** Build a complete language model that generates text — using everything from notebooks 01, 02, and 03.

### What we build:
```
Input word → Embedding → Attention → FeedForward → Output probabilities → Next word
```

### How GPT works (in one sentence):
Given the words so far, predict the most likely next word. Repeat.

```
'deep'          → predicts → 'learning'
'deep learning' → predicts → 'is'
'deep learning is' → predicts → 'amazing'
```

---
- Notebook 01: neural network و backprop
- Notebook 02: tokenization و embedding
- Notebook 03: attention mechanism
- Notebook 04: together = mini-GPT

In [2]:
import numpy as np
np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)
print('Libraries loaded ✓')

Libraries loaded ✓


---
## Part 1 — Training Data

GPT learns by predicting the next word.
We give it sentences, and it learns: given this word, what comes next?

In [1]:
# ── Training Data ─────────────────────────────────────────

corpus = [
    'i love ai',
    'i love deep learning',
    'deep learning is amazing',
    'ai is the future',
    'i love the future',
]

# Build vocabulary
all_words = []
for s in corpus:
    all_words.extend(s.split())

vocab     = sorted(set(all_words))
vocab_size = len(vocab)
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for w, i in w2i.items()}

print(f'Vocabulary ({vocab_size} words): {vocab}')
print(f'word_to_idx: {w2i}')

# Build training pairs: (input word, target next word)
pairs = []
for s in corpus:
    words = s.split()
    for i in range(len(words) - 1):
        pairs.append((w2i[words[i]], w2i[words[i+1]]))

print(f'\nTraining pairs (input → target):')
for inp, tgt in pairs[:6]:
    print(f"  '{i2w[inp]}' → '{i2w[tgt]}'")
print(f'  ... ({len(pairs)} pairs total)')

Vocabulary (9 words): ['ai', 'amazing', 'deep', 'future', 'i', 'is', 'learning', 'love', 'the']
word_to_idx: {'ai': 0, 'amazing': 1, 'deep': 2, 'future': 3, 'i': 4, 'is': 5, 'learning': 6, 'love': 7, 'the': 8}

Training pairs (input → target):
  'i' → 'love'
  'love' → 'ai'
  'i' → 'love'
  'love' → 'deep'
  'deep' → 'learning'
  'deep' → 'learning'
  ... (14 pairs total)


---
## Part 2 — Model Architecture

```
Input (word index)
    ↓
Embedding Layer      word → dense vector
    ↓
Attention Layer      understand context
    ↓
Feed Forward Layer   process further
    ↓
Output Layer         scores for each word
    ↓
Softmax              probabilities → pick next word
```

In [3]:
# ── Model Architecture ────────────────────────────────────

d_model = 8    # embedding size
d_k     = 4    # attention Q/K/V size
d_ff    = 16   # feed forward hidden size

# ── Embedding weights ──
E = np.random.randn(vocab_size, d_model) * 0.1

# ── Attention weights ──
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_k) * 0.1
W_O = np.random.randn(d_k, d_model) * 0.1    # project back to d_model

# ── Feed Forward weights ──
W1_ff = np.random.randn(d_model, d_ff) * 0.1
b1_ff = np.zeros(d_ff)
W2_ff = np.random.randn(d_ff, d_model) * 0.1
b2_ff = np.zeros(d_model)

# ── Output weights ──
W_out = np.random.randn(d_model, vocab_size) * 0.1
b_out = np.zeros(vocab_size)

print('Model weights initialized:')
print(f'  Embedding:    {E.shape}        ← {vocab_size} words × {d_model} dims')
print(f'  Attention:    W_Q{W_Q.shape} W_K{W_K.shape} W_V{W_V.shape}')
print(f'  FeedForward:  W1{W1_ff.shape} W2{W2_ff.shape}')
print(f'  Output:       {W_out.shape}   ← {d_model} dims → {vocab_size} words')

Model weights initialized:
  Embedding:    (9, 8)        ← 9 words × 8 dims
  Attention:    W_Q(8, 4) W_K(8, 4) W_V(8, 4)
  FeedForward:  W1(8, 16) W2(16, 8)
  Output:       (8, 9)   ← 8 dims → 9 words


---
## Part 3 — Helper Functions

In [4]:
# ── Helper Functions ──────────────────────────────────────

def softmax(x):
    """Scores → probabilities (all positive, sum to 1)"""
    e = np.exp(x - np.max(x))
    return e / e.sum()

def relu(x):
    """Negative → 0, Positive → keep"""
    return np.maximum(0, x)

print('Helper functions defined ✓')
print(f"  softmax([1,2,3]) = {softmax(np.array([1,2,3]))}")
print(f"  relu([-2,0,3])   = {relu(np.array([-2,0,3]))}")

Helper functions defined ✓
  softmax([1,2,3]) = [0.09  0.245 0.665]
  relu([-2,0,3])   = [0 0 3]


---
## Part 4 — Forward Pass

Given one word, predict the next word.
This is the same as notebook 03 (attention) + a feedforward layer + output.


In [5]:
# ── Forward Pass ──────────────────────────────────────────

def forward(word_idx):
    """
    Given a word index, predict probability of each next word.

    Steps:
      1. Embedding lookup    (notebook 02)
      2. Self-Attention      (notebook 03)
      3. Feed Forward        (notebook 01 - neural network)
      4. Output projection   → score for each word
      5. Softmax             → probabilities
    """
    # Step 1: Embedding lookup
    x = E[word_idx]                        # shape: (d_model,)

    # Step 2: Self-Attention (simplified for single token)
    q = x @ W_Q                            # Query
    k = x @ W_K                            # Key
    v = x @ W_V                            # Value
    score = np.dot(q, k) / np.sqrt(d_k)   # attention score
    attn_out = v * score                   # weighted value
    attn_out = attn_out @ W_O             # project back
    x = x + attn_out                      # residual connection

    # Step 3: Feed Forward Network
    ff1 = relu(x @ W1_ff + b1_ff)        # hidden layer
    ff2 = ff1 @ W2_ff + b2_ff             # output layer
    x = x + ff2                           # residual connection

    # Step 4: Output projection
    logits = x @ W_out + b_out            # shape: (vocab_size,)

    # Step 5: Softmax
    probs = softmax(logits)               # shape: (vocab_size,)

    return probs


# Test forward pass before training
test_word = 'deep'
probs = forward(w2i[test_word])
print(f"Before training — predictions for '{test_word}':")
top3 = np.argsort(probs)[::-1][:3]
for idx in top3:
    print(f"  '{i2w[idx]}': {probs[idx]:.3f}")
print('(random predictions — model has not learned yet)')

Before training — predictions for 'deep':
  'amazing': 0.116
  'future': 0.115
  'deep': 0.115
(random predictions — model has not learned yet)


---
## Part 5 — Training

Same training loop as notebook 01!
- Forward pass → get prediction
- Loss → how wrong?
- Update weights → get better

Loss function: **Cross-Entropy** = `-log(probability of correct word)`
- If model predicts correctly with 99% → loss ≈ 0.01 (good)
- If model predicts correctly with 1% → loss ≈ 4.6 (bad)


In [6]:
# ── Training Loop ─────────────────────────────────────────

lr = 0.05
print('=== Training mini-GPT ===\n')
print(f'  {"Epoch":<8} {"Loss":<12}')
print('  ' + '─' * 22)

for epoch in range(500):
    total_loss = 0

    for inp_idx, tgt_idx in pairs:

        # Forward pass
        probs = forward(inp_idx)

        # Cross-entropy loss: -log(probability of correct word)
        loss = -np.log(probs[tgt_idx] + 1e-9)
        total_loss += loss

        # Simplified gradient update on output weights
        # (full backprop through all layers)
        d_logits = probs.copy()
        d_logits[tgt_idx] -= 1.0     # gradient of cross-entropy + softmax

        # Recompute hidden state for gradient
        x = E[inp_idx]
        q = x @ W_Q
        k = x @ W_K
        v = x @ W_V
        attn = v * (np.dot(q, k) / np.sqrt(d_k))
        attn_proj = attn @ W_O
        x_res = x + attn_proj
        ff1 = relu(x_res @ W1_ff + b1_ff)
        ff2 = ff1 @ W2_ff + b2_ff
        h = x_res + ff2

        # Update output weights
        W_out -= lr * np.outer(h, d_logits)
        b_out -= lr * d_logits

        # Update embedding
        d_h = d_logits @ W_out.T
        E[inp_idx] -= lr * d_h * 0.1

    avg_loss = total_loss / len(pairs)
    if epoch % 100 == 0:
        print(f'  {epoch:<8} {avg_loss:<12.4f}')

print('\nTraining complete ✓')

=== Training mini-GPT ===

  Epoch    Loss        
  ──────────────────────
  0        2.2107      
  100      0.5832      
  200      0.3853      
  300      0.3676      
  400      0.3631      

Training complete ✓


---
## Part 6 — Test: Next Word Prediction

In [7]:
# ── Test: Next Word Prediction ────────────────────────────

print('=== Predictions after training ===\n')

test_words = ['i', 'deep', 'ai', 'love']

for word in test_words:
    probs = forward(w2i[word])
    top3  = np.argsort(probs)[::-1][:3]

    print(f"  Input: '{word}'")
    for rank, idx in enumerate(top3):
        bar = '█' * int(probs[idx] * 25)
        print(f"    {rank+1}. '{i2w[idx]}'{'':.<12} {probs[idx]:.3f}  {bar}")
    print()

=== Predictions after training ===

  Input: 'i'
    1. 'love'............ 0.997  ████████████████████████
    2. 'amazing'............ 0.001  
    3. 'learning'............ 0.001  

  Input: 'deep'
    1. 'learning'............ 0.995  ████████████████████████
    2. 'love'............ 0.001  
    3. 'the'............ 0.001  

  Input: 'ai'
    1. 'is'............ 0.991  ████████████████████████
    2. 'future'............ 0.002  
    3. 'the'............ 0.002  

  Input: 'love'
    1. 'the'............ 0.356  ████████
    2. 'deep'............ 0.328  ████████
    3. 'ai'............ 0.310  ███████



---
## Part 7 — Text Generation

Now we can generate full sentences — just keep predicting the next word!

**Temperature** controls creativity:
- Low (0.5) = safe and predictable
- High (1.5) = creative and surprising

In [8]:
# ── Text Generation ───────────────────────────────────────

def generate(start_word, max_words=5, temperature=0.8):
    """
    Generate text starting from a given word.

    Args:
        start_word:  first word
        max_words:   how many words to generate
        temperature: creativity (0.5=safe, 1.5=creative)
    """
    result  = [start_word]
    current = w2i[start_word]

    for _ in range(max_words - 1):
        # Get probability distribution
        probs = forward(current)

        # Apply temperature
        # Low temperature → more confident
        # High temperature → more random
        probs = np.power(probs, 1.0 / temperature)
        probs = probs / probs.sum()

        # Sample from distribution
        next_idx  = np.random.choice(len(vocab), p=probs)
        next_word = i2w[next_idx]

        result.append(next_word)
        current = next_idx

    return ' '.join(result)


print('=== Text Generation ===\n')
np.random.seed(10)

examples = [
    ('i',    0.5,  'conservative'),
    ('deep', 0.5,  'conservative'),
    ('i',    0.8,  'balanced'),
    ('ai',   1.0,  'creative'),
    ('i',    1.5,  'very creative'),
]

for start, temp, label in examples:
    sentence = generate(start, max_words=5, temperature=temp)
    print(f"  start='{start}'  temp={temp} ({label})")
    print(f"  → '{sentence}'")
    print()

=== Text Generation ===

  start='i'  temp=0.5 (conservative)
  → 'i love ai is the'

  start='deep'  temp=0.5 (conservative)
  → 'deep learning is amazing the'

  start='i'  temp=0.8 (balanced)
  → 'i love ai is the'

  start='ai'  temp=1.0 (creative)
  → 'ai future is the future'

  start='i'  temp=1.5 (very creative)
  → 'i love ai is the'



---
## Part 8 — Compare: Before vs After Training

In [9]:
# ── What the model learned ────────────────────────────────

print('=== What the model learned ===\n')
print('Training data patterns:')
for s in corpus:
    words = s.split()
    for i in range(len(words)-1):
        print(f"  '{words[i]}' → '{words[i+1]}'")

print('\nModel predictions (top 1):')
for word in ['i', 'deep', 'ai', 'love', 'learning', 'is']:
    if word in w2i:
        probs = forward(w2i[word])
        best  = i2w[np.argmax(probs)]
        conf  = np.max(probs)
        print(f"  '{word}' → '{best}'  ({conf:.1%} confident)")

=== What the model learned ===

Training data patterns:
  'i' → 'love'
  'love' → 'ai'
  'i' → 'love'
  'love' → 'deep'
  'deep' → 'learning'
  'deep' → 'learning'
  'learning' → 'is'
  'is' → 'amazing'
  'ai' → 'is'
  'is' → 'the'
  'the' → 'future'
  'i' → 'love'
  'love' → 'the'
  'the' → 'future'

Model predictions (top 1):
  'i' → 'love'  (99.7% confident)
  'deep' → 'learning'  (99.5% confident)
  'ai' → 'is'  (99.1% confident)
  'love' → 'the'  (35.6% confident)
  'learning' → 'is'  (99.2% confident)
  'is' → 'the'  (52.7% confident)


---
## Summary

We built a complete language model from scratch that:
- Learns from text data (training)
- Predicts the next word given any input
- Generates coherent sentences
- Uses temperature to control creativity

The architecture: **Embedding → Attention → FeedForward → Output → Softmax**

This is exactly what GPT-2, GPT-4, and Claude do — just with:
- 50,257 words instead of 9
- 768–12,288 dimensions instead of 8
- 12–96 layers instead of 1
- Billions of parameters instead of hundreds

---

| | mini-GPT ما | GPT-2 | GPT-4 |
|---|---|---|---|
| Vocabulary | 9 | 50,257 | 100,000+ |
| Embedding dim | 8 | 768 | 12,288 |
| Layers | 1 | 12 | 96+ |
| Parameters | ~200 | 124M | ~1.8T |

---
**Next:** `05_huggingface.ipynb` — use the real GPT-2 with HuggingFace Transformers